# Data Mining - A.A. 2025/2026
## Progetto Gruppo G3.3
### Membri:
- Masala Gabriele: 60/61/66245 (60/99/00004)
- Mantega Gabriele: 60/61/66251 (60/99/00006)
- Aresu Matteo: 60/99/00014

# 1 - Analisi del Dataset

In [3]:
# Dichiarazione Import
import pandas as pd
import numpy as np
import os
from sklearn import datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sktext

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [4]:
# Direttive di Pandas per avere un output più leggibile
pd.set_option('display.max_colwidth', 100)

# Caricamento DataFrame da sklearn
categories = ['rec.autos', 'rec.sport.baseball', 'sci.electronics', 'sci.med', 'sci.space']
train_dataset = datasets.fetch_20newsgroups(subset='train', categories=categories)
test_dataset = datasets.fetch_20newsgroups(subset='test', categories=categories)
X_train = pd.DataFrame(train_dataset.data).rename(columns={0: 'raw_text'})
y_train = pd.DataFrame(train_dataset.target).rename(columns={0: 'category'})

# Stampo la descrizione del dataset
#print(train_dataset.DESCR[90:845])

In [5]:
# Stampo il Training Set
#display(X_train)

# Stampo il Test Set
#display(y_train)

In [6]:
# Stampo a schermo il primo record del Dataset
#print(X_train['raw_text'][0])

# 2 - Preprocessing

In [7]:
# FUNZIONI DI PREPROCESSING

# Load GloVe embeddings into a dictionary (presa da spot-intelligence)
def load_embeddings(file_path):
    embeddings = {}
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found. Please check the path and try again.")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

# ---

# esegue: lowercase, tokenizzazione, rimozione della punteggiatura e delle stopword
def preprocess_text(text):
    # Converti il testo in minuscolo
    text = text.lower()
    
    # Rimuovi la punteggiatura
    text = ''.join(char for char in text if char.isalnum() or char.isspace())
    
    # Tokenizza il testo
    tokens = text.split()
    
    # Rimuovi le stopword (opzionale, puoi usare una lista di stopword)
    stopwords = sktext.ENGLISH_STOP_WORDS # stopwords di sklearn
    tokens = [tkn for tkn in tokens if tkn not in stopwords]
    
    return tokens

# ---

# Calcola l'embedding del documento come media degli embedding delle parole
def get_document_embedding(tokens, glove_embeddings):
    # Ottieni gli embedding per ogni parola nel documento
    word_embeddings = [glove_embeddings[token] for token in tokens if token in glove_embeddings.keys()] # Verifica se la parola ha un embedding disponibile
    
    # Calcola la media degli embedding per ottenere l'embedding del documento
    if word_embeddings:
        document_embedding = np.mean(word_embeddings, axis=0) 
    else:
        document_embedding = np.zeros(len(word_embeddings[0]))  # Se nessuna parola ha embedding, restituisci un vettore di zeri
    
    return document_embedding

# ---

# ...


## 2.1 Tf-Idf

In [31]:

# Utilizzo la classe TfidfVectorizer per convertire i record testuali in record numerici
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
tfidf_vectorized_data = tfidf_vectorizer.fit_transform(X_train['raw_text'])

tfidf_train = np.array(pd.DataFrame(tfidf_vectorized_data.toarray(), columns=tfidf_vectorizer.get_feature_names_out()))

## 2.2 Word Embeddings

In [14]:
# caricamento degli embeddings GloVe
glove_embeddings_path = 'glove.6B.100d.txt'  
glove_embeddings = load_embeddings(glove_embeddings_path)

# Preprocessamento del testo e calcolo degli embedding dei documenti
rows = []
for text in X_train['raw_text']:
    tokens = preprocess_text(text)
    rows.append(get_document_embedding(tokens, glove_embeddings))

# Creazione di un DataFrame per visualizzare i risultati
preprocessed_data = pd.DataFrame({'preprocessed_text':rows})
#display(preprocessed_data)

,preprocessed_text
0,"[0.013487218, 0.19742657, 0.15495875, -0.04091889, -0.030088535, -0.04530107, -0.014340127, -0.0..."
1,"[0.032147814, 0.18368375, 0.228122, -0.38081115, -0.14968252, 0.057901923, 0.15478352, -0.112096..."
2,"[-0.08363266, 0.150303, 0.3050858, -0.28667924, 0.12461482, 0.011300105, 0.03869984, 0.010601276..."
3,"[-0.15064834, 0.2542557, 0.36458164, -0.27481288, 0.04217179, -0.005881374, 0.1582556, 0.0441756..."
4,"[-0.1191443, 0.16003343, 0.1168153, -0.09900364, 0.082644336, -0.13636848, -0.007913775, -0.0538..."
...,...
2964,"[-0.077621244, 0.13963628, 0.05995103, -0.21984927, 0.13088726, 0.04048803, -0.03347907, -0.0985..."
2965,"[-0.08538793, 0.2544477, 0.22309266, -0.28458503, -0.12043153, 0.033386722, -0.15904091, -0.0061..."
2966,"[-0.12544757, 0.14032832, 0.2988885, -0.18858485, -0.026737928, 0.018732315, 0.19242465, 0.17730..."
2967,"[-0.13504371, 0.29273108, 0.18721282, -0.062121704, -0.06964278, -0.16738385, 0.052568115, 0.114..."


# 3. Classificazione

In [24]:
# Inizializzazione classificataori (#todo parametri sparati, dobbiamo migliorarli)
classifiers = {
    'Decision Tree': DecisionTreeClassifier(max_depth=15, min_samples_split=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}


In [ ]:
# Alleniamo e valutiamo tutti i modelli
results = {}
for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    
    # Allenamento modello con TF-IDF
    clf.fit(tfidf_vectorized_data, y_train)
    '''
    y_pred = clf.predict(X_val)
    
    results[name] = {
    'accuracy': accuracy_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_val, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_val, y_pred, average='weighted', zero_division=0)
    }
    print(f"{name} - Accuracy: {results[name]['accuracy']:.4f}")
    
# Display results
results_df = pd.DataFrame(results).T
print("\n" + "="*70)
print("Classification Results Summary")
print("="*70)
print(results_df.round(4))
    '''


Training Decision Tree...

Training KNN...

Training MLP...


/home/matthew/.local/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/home/matthew/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1223: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/matthew/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/home/matthew/.local/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fi


Training Random Forest...

Training AdaBoost...


/home/matthew/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



Training XGBoost...
